In [1]:
# Configuración
import findspark
findspark.init()

from pyspark.sql import SparkSession

In [ ]:
# Crear una sesión de Spark
spark = SparkSession.builder \
    .appName("AnalisisVentas") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# Leer CSV 
df = spark.read.csv("../data/online_retail.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5, truncate=False)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+-----------------------------------+--------+------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/10 8:26|2,55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |12/1/10 8:26|3,39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |12/1/10 8:26|2,7

In [3]:
# Número de filas
print(f"Total de registros: {df.count()}")

# Número de columnas
print(f"Total de columnas: {len(df.columns)}")

# Lista de columnas
print(df.columns)


Total de registros: 541909
Total de columnas: 8
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [ ]:
df.describe().show() # estadísticas descriptivas básicas
# el .show() es para mostrar el resultado en consola de dataframe

+-------+------------------+------------------+--------------------+------------------+-------------+------------------+------------------+-----------+
|summary|         InvoiceNo|         StockCode|         Description|          Quantity|  InvoiceDate|         UnitPrice|        CustomerID|    Country|
+-------+------------------+------------------+--------------------+------------------+-------------+------------------+------------------+-----------+
|  count|            541909|            541909|              540455|            541909|       541909|            541909|            406829|     541909|
|   mean|  559965.752026781|27623.240210938104|             20713.0|  9.55224954743324|         NULL|29.921163668665333|15287.690570239585|       NULL|
| stddev|13428.417280796919| 16799.73762842769|                NULL|218.08115785023327|         NULL| 595.7455525989112| 1713.600303321597|       NULL|
|    min|            536365|             10002| 4 PURPLE FLOCK D...|            -80995|1

In [12]:
from pyspark.sql.functions import col, count, sum, avg

num_paises = df.select("Country").distinct().count()
print(f"Número de países distintos: {num_paises}")


df.groupBy("Country").count().orderBy(col("count").desc()).show(5)

Número de países distintos: 38
+--------------+------+
|       Country| count|
+--------------+------+
|United Kingdom|495478|
|       Germany|  9495|
|        France|  8557|
|          EIRE|  8196|
|         Spain|  2533|
+--------------+------+
only showing top 5 rows


In [16]:
df.groupBy("Country").count()\
    .orderBy(col("count")\
    .desc())\
    .show(10)

+--------------+------+
|       Country| count|
+--------------+------+
|United Kingdom|495478|
|       Germany|  9495|
|        France|  8557|
|          EIRE|  8196|
|         Spain|  2533|
|   Netherlands|  2371|
|       Belgium|  2069|
|   Switzerland|  2002|
|      Portugal|  1519|
|     Australia|  1259|
+--------------+------+
only showing top 10 rows
